In [1]:
import glob
import logging
import os
import warnings
from time import time

import h5py
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed
from scipy.optimize import OptimizeWarning, minimize
from scipy.optimize._numdiff import approx_derivative
from statsmodels.robust import scale
from statsmodels.robust.norms import RobustNorm, TukeyBiweight

In [2]:
%matplotlib inline

## Function definitions

In [3]:
from dev import single_exp, double_exp, bright, single_exp_jac, double_exp_jac, bright_jac
from dev import single_exp_with_jac, double_exp_with_jac, bright_with_jac
from dev import single_exp_jax, double_exp_jax, bright_jax, TukeyBiweight_jax
from dev import tc_brightfit, fit_baseline_trend, fit_baseline_trend2

#### load data 

In [4]:
def get_valid_corrected_f(path):
    dr = path.split("/")[-1]
    data = []
    with h5py.File(f"{path}/{dr}_data.h5") as f:
        for k in f['planes'].keys():
            data.append(f[f'planes/{k}/corrected_f'][f[f'planes/{k}/valid_roi_inds'][:]])
    return data

## Data 

In [5]:
dirs = glob.glob("/data/test-dataset-for-dff_multiplane-ophys_02/*/*")
dirs

['/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/804670',
 '/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/775682',
 '/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/782149',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/753562',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/729088',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/758265',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/753561',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/724567',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/755212',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/759075',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/726433',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/747443',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi1/757436',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi1/69

In [6]:
traces = get_valid_corrected_f(dirs[0])

In [7]:
[t.shape for t in traces]

[(56, 48374),
 (46, 48374),
 (73, 48374),
 (73, 48374),
 (62, 48374),
 (56, 48374),
 (41, 48374),
 (49, 48374),
 (59, 48403),
 (46, 48403),
 (71, 48403),
 (75, 48403),
 (65, 48403),
 (57, 48403),
 (39, 48403),
 (53, 48403),
 (57, 48427),
 (48, 48427),
 (67, 48427),
 (73, 48427),
 (65, 48427),
 (61, 48427),
 (41, 48427),
 (48, 48427),
 (55, 48390),
 (46, 48390),
 (65, 48390),
 (76, 48390),
 (63, 48390),
 (56, 48390),
 (40, 48390),
 (48, 48390),
 (60, 48373),
 (42, 48373),
 (65, 48373),
 (76, 48373),
 (66, 48373),
 (58, 48373),
 (40, 48373),
 (52, 48373),
 (58, 48366),
 (44, 48366),
 (66, 48366),
 (75, 48366),
 (61, 48366),
 (54, 48366),
 (42, 48366),
 (51, 48366)]

In [8]:
trace = traces[0][0]

In [9]:
frame_rate = 10.63  # looked up manually from session.json of multiplane-ophys_804670_2025-09-24_09-30-56_processed_2025-10-10_22-28-44
timestamps = np.arange(len(trace)) / frame_rate

In [10]:
single_exp_init = [trace[-1000:].mean(), 0.35, 3600]
double_exp_init = [trace[-1000:].mean(), 0.35, 0.2, 3600, 240]
bright_init = [trace[-1000:].mean(), 0.35, 0.2, 0.1, 0.1, 3600, 240, 50, 2000]

single_exp_bounds = [(0, np.inf)] * 2 + [(300, np.inf)]
double_exp_bounds = [(0, np.inf)] * 3 + [(300, np.inf), (1, 1200)]
bright_bounds = [(0, np.inf)] * 5 + [(300, np.inf), (1, 1200), (1, 180), (60, np.inf)]

## check analytical Jacobians 

In [11]:
for model, jac, start_params in (
    (single_exp, single_exp_jac, single_exp_init),
    (double_exp, double_exp_jac, double_exp_init),
    (bright, bright_jac, bright_init),
):
    J_num = approx_derivative(lambda p: model(p, timestamps), start_params)
    J_ana = jac(start_params, timestamps)
    print(np.max(np.abs(J_num - J_ana)))  # should be ~1e-8

3.843035756290192e-08
4.878393156104721e-08
5.2436689657042734e-08


In [12]:
for model, jac, start_params in (
    (single_exp, single_exp_with_jac, single_exp_init),
    (double_exp, double_exp_with_jac, double_exp_init),
    (bright, bright_with_jac, bright_init),
):
    J_num = approx_derivative(lambda p: model(p, timestamps), start_params)
    J_ana = jac(start_params, timestamps, True)[1]
    print(np.max(np.abs(J_num - J_ana)))  # should be ~1e-8

3.843035756290192e-08
4.878393156104721e-08
5.2436689657042734e-08


## unbounded OLS 

In [13]:
for model, jac, start_params in (
    (single_exp, single_exp_jac, single_exp_init),
    (double_exp, double_exp_jac, double_exp_init),
):
    print("\n" + model.__name__)
    for optimizer in ("Nelder-Mead", "BFGS", "L-BFGS-B", "CG", "Newton-CG", "TNC", "SLSQP", "trust-constr"):
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend(trace, timestamps, model, start_params,
                                          jac=None if optimizer=="Nelder-Mead" else jac,
                                          optimizer=optimizer)
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=200):
            # print(f"{optimizer:13}  Time={tic:7.4f}s  Loss={((F0-trace)**2).mean()/2:.2f}", res.x, res.message)
            print(f"{optimizer:13}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)


single_exp
Nelder-Mead    Time= 0.1497s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Optimization terminated successfully.
BFGS           Time= 0.0315s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Desired error not necessarily achieved due to precision loss.
L-BFGS-B       Time= 0.0223s  Loss=36881.96 [ 1.22e+03 -2.47e-01  3.60e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
CG             Time= 0.5377s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.35e+03] Desired error not necessarily achieved due to precision loss.
Newton-CG      Time= 0.2149s  Loss=36882.06 [ 1.22e+03 -2.46e-01  3.60e+03] Optimization terminated successfully.
TNC            Time= 0.0821s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Converged (|x_n-x_(n-1)| ~= 0)
SLSQP          Time= 0.2699s  Loss=38781.41 [ 1.05e+03 -6.73e-01  3.68e-03] Optimization terminated successfully
trust-constr   Time= 0.1685s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] `xtol` termination condition is satisfied.

double_exp
Neld

In [14]:
for model, jac, start_params in (
    (single_exp, single_exp_jac, single_exp_init),
    (double_exp, double_exp_jac, double_exp_init),
    (bright, bright_jac, bright_init),
):
    print("\n" + model.__name__)
    for method in ("BFGS_noJac", "BFGS", "L-BFGS-B_noJac", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend(trace, timestamps, model, start_params, jac=None if method[-5:]=="noJac" else jac, optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)


single_exp
BFGS_noJac      Time= 0.0681s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Desired error not necessarily achieved due to precision loss.
BFGS            Time= 0.0302s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Desired error not necessarily achieved due to precision loss.
L-BFGS-B_noJac  Time= 0.0437s  Loss=36881.96 [ 1.22e+03 -2.47e-01  3.60e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
L-BFGS-B        Time= 0.0213s  Loss=36881.96 [ 1.22e+03 -2.47e-01  3.60e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp
BFGS_noJac      Time= 1.7136s  Loss=30638.93 [1135.14 -100.98  101.39  458.72  453.49] Desired error not necessarily achieved due to precision loss.
BFGS            Time= 2.8107s  Loss=30638.90 [1135.14 -603.    603.4   456.56  455.68] Desired error not necessarily achieved due to precision loss.
L-BFGS-B_noJac  Time= 7.9557s  Loss=30638.93 [1135.15 -101.93  102.33  458.74  453.56] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


#### compute model and jac together to share intermediate results 

In [15]:
for model, start_params in (
    (single_exp_with_jac, single_exp_init),
    (double_exp_with_jac, double_exp_init),
    (bright_with_jac, bright_init),
):
    print("\n" + model.__name__)
    for method in ("BFGS", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend2(trace, timestamps, model, start_params, optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)


single_exp_with_jac
BFGS            Time= 0.0284s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time= 0.0684s  Loss=36881.96 [ 1.22e+03 -2.47e-01  3.60e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_with_jac
BFGS            Time= 2.3987s  Loss=30638.90 [1135.14 -603.    603.4   456.56  455.68] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time=11.6313s  Loss=30638.90 [ 1135.14 -1182.01  1182.41   456.34   455.9 ] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_with_jac
BFGS            Time=14.5002s  Loss=29454.06 [ 3.61e+02  1.35e+06 -1.24e+07  2.18e+07  1.00e+00  2.98e+03  1.13e+02  1.38e+02  4.88e+08] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time= 3.4057s  Loss=29694.05 [ 7.20e+02  2.17e+00  6.62e+04 -6.62e+04  9.41e-01  3.58e+03  1.22e+02  1.22e+02  1.86e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FA

#### JAX autograd

In [16]:
for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    for model, start_params in (
        (single_exp_jax, single_exp_init),
        (double_exp_jax, double_exp_init),
        (bright_jax, bright_init),
    ):
        print("\n" + model.__name__)
        for optimizer in ("BFGS", "L-BFGS-B"):
            tic = -time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                F0, res = fit_baseline_trend(trace, timestamps, model, start_params, jac="jax", optimizer=optimizer,
                                           optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10),
                                           dtype=dtype)
            tic += time()
            with np.printoptions(precision=2, suppress=False, linewidth=120):
                print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)



float64

single_exp_jax
BFGS            Time= 0.4967s  Loss=36877.38 [ 1.26e+03 -2.65e-01  4.36e+03] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time= 0.1259s  Loss=36881.96 [ 1.22e+03 -2.47e-01  3.60e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_jax
BFGS            Time= 1.4277s  Loss=30638.90 [1135.14 -497.31  497.71  456.65  455.59] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time= 5.1570s  Loss=30638.90 [1135.14 -556.26  556.67  456.57  455.62] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
BFGS            Time= 2.7824s  Loss=29465.62 [ 4.22e+02  1.17e+05 -9.32e+05  1.52e+06  1.00e+00  3.07e+03  9.96e+01  1.35e+02  5.20e+07] Desired error not necessarily achieved due to precision loss.
L-BFGS-B        Time= 2.8117s  Loss=29564.33 [ 5.44e+02  3.56e+00  8.24e+04 -8.24e+04  6.76e-01  3.57e+03  4.93e+02  4.93e+02  1.82e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

## bounded OLS 

In [17]:
for model, jac, start_params, bounds in (
    (single_exp, single_exp_jac, single_exp_init, single_exp_bounds),
    (double_exp, double_exp_jac, double_exp_init, double_exp_bounds),
    (bright, bright_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for method in ("Nelder-Mead", "L-BFGS-B_noJac", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend(trace, timestamps, model, start_params, bounds=bounds,
                                          jac=jac if method=="L-BFGS-B" else None,
                                          optimizer=optimizer,
                                          optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10) if optimizer=="L-BFGS-B" else None)
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)


single_exp
Nelder-Mead     Time= 0.1311s  Loss=38786.56 [ 1049.25     0.   11951.34] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 0.0196s  Loss=38786.56 [1049.25    0.   3485.95] CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
L-BFGS-B        Time= 0.0211s  Loss=38786.56 [1049.25    0.   3485.95] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp
Nelder-Mead     Time= 0.4392s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.00e+02 9.70e+01] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 0.1784s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.37e+03 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
L-BFGS-B        Time= 0.2052s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.37e+03 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright
Nelder-Mead     Time=26.2539s  Loss=29459.96 [7.31e+02 2.40e+03 2.04e+04 1.82e+02 1.00e+00 2.96e+03 1.58e+02 1.00e+00 1.72e+06] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 8.11

#### compute model and jac together to share intermediate results 

In [18]:
for model, start_params, bounds in (
    (single_exp_with_jac, single_exp_init, single_exp_bounds),
    (double_exp_with_jac, double_exp_init, double_exp_bounds),
    (bright_with_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for method in ("L-BFGS-B",):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend2(trace, timestamps, model, start_params, bounds=bounds, optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)


single_exp_with_jac
L-BFGS-B        Time= 0.0182s  Loss=38786.56 [1049.25    0.   3485.95] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_with_jac
L-BFGS-B        Time= 0.1892s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.37e+03 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_with_jac
L-BFGS-B        Time= 3.3133s  Loss=29679.59 [6.87e+02 2.52e+00 3.29e+01 5.10e-02 9.50e-01 3.58e+03 1.47e+02 1.29e+02 1.97e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


#### JAX autograd

In [19]:
for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    for model, start_params, bounds in (
        (single_exp_jax, single_exp_init, single_exp_bounds),
        (double_exp_jax, double_exp_init, double_exp_bounds),
        (bright_jax, bright_init, bright_bounds),
    ):
        print("\n" + model.__name__)
        for optimizer in ("L-BFGS-B",):
            tic = -time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                F0, res = fit_baseline_trend(trace, timestamps, model, start_params, bounds=bounds, jac="jax", optimizer=optimizer,
                                           optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10),
                                           dtype=dtype)
            tic += time()
            with np.printoptions(precision=2, suppress=False, linewidth=120):
                print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/2/trace.size:.2f}", res.x, res.message)



float64

single_exp_jax
L-BFGS-B        Time= 0.2086s  Loss=38786.56 [1049.25    0.   3485.95] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_jax
L-BFGS-B        Time= 0.2713s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.37e+03 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
L-BFGS-B        Time= 5.1203s  Loss=29482.15 [1.78e+01 1.98e+02 1.42e+03 3.33e+02 9.63e-01 4.71e+03 1.61e+02 1.46e+02 3.31e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


float32

single_exp_jax
L-BFGS-B        Time= 0.2695s  Loss=38786.56 [1049.25    0.   3485.95] ABNORMAL: 

double_exp_jax
L-BFGS-B        Time= 0.2936s  Loss=37572.87 [1.04e+03 0.00e+00 4.69e-01 3.37e+03 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
L-BFGS-B        Time= 0.4282s  Loss=31495.18 [1.09e+03 5.70e-01 3.77e+00 2.86e-01 7.60e-01 3.60e+03 2.40e+02 5.00e+01 2.00e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


## bounded robust regression (Tukey) 

In [20]:
for model, jac, start_params, bounds in (
    (single_exp, single_exp_jac, single_exp_init, single_exp_bounds),
    (double_exp, double_exp_jac, double_exp_init, double_exp_bounds),
    (bright, bright_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for method in ("Nelder-Mead", "L-BFGS-B_noJac", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend(trace, timestamps, model, start_params, bounds=bounds,
                                         M=TukeyBiweight(3),
                                         jac=jac if method=="L-BFGS-B" else None,
                                         optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10) if optimizer=="L-BFGS-B" else None)
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)


single_exp
Nelder-Mead     Time= 0.7598s  Loss=29155.95 [ 1030.81     0.   18031.9 ] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 0.0738s  Loss=29160.82 [1030.83    0.   3599.02] CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
L-BFGS-B        Time= 0.0598s  Loss=29160.84 [1030.83    0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp
Nelder-Mead     Time= 2.0507s  Loss=27751.63 [1.02e+03 4.72e-15 5.00e-01 1.92e+04 1.03e+02] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 1.7847s  Loss=27753.58 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
L-BFGS-B        Time= 0.5886s  Loss=27753.59 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright
Nelder-Mead     Time=52.5888s  Loss=21633.56 [3.61e+02 3.18e+04 1.65e+04 2.82e+05 1.00e+00 2.96e+03 2.65e+02 1.47e+02 1.15e+07] Optimization terminated successfully.
L-BFGS-B_noJac  Time=15.00

#### compute model and jac together to share intermediate results 

In [21]:
for model, start_params, bounds in (
    (single_exp_with_jac, single_exp_init, single_exp_bounds),
    (double_exp_with_jac, double_exp_init, double_exp_bounds),
    (bright_with_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for method in ("L-BFGS-B",):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = fit_baseline_trend2(trace, timestamps, model, start_params, bounds=bounds, optimizer=optimizer,
                                          M=TukeyBiweight(3),
                                          optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)


single_exp_with_jac
L-BFGS-B        Time= 0.1051s  Loss=29160.84 [1030.83    0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_with_jac
L-BFGS-B        Time= 0.6711s  Loss=27753.59 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_with_jac
L-BFGS-B        Time=13.1126s  Loss=21595.90 [6.96e+01 5.18e+01 8.08e+02 1.41e+03 9.91e-01 4.28e+03 1.40e+02 2.09e+01 3.21e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


#### JAX autograd

In [22]:
for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    for model, start_params, bounds in (
        (single_exp_jax, single_exp_init, single_exp_bounds),
        (double_exp_jax, double_exp_init, double_exp_bounds),
        (bright_jax, bright_init, bright_bounds),
    ):
        print("\n" + model.__name__)
        for optimizer in ("L-BFGS-B",):
            tic = -time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                F0, res = fit_baseline_trend(trace, timestamps, model, start_params, bounds=bounds, jac="jax", optimizer=optimizer,
                                           M=TukeyBiweight_jax(3),
                                           optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10),
                                           dtype=dtype)
            tic += time()
            with np.printoptions(precision=2, suppress=False, linewidth=120):
                print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)



float64

single_exp_jax
L-BFGS-B        Time= 0.7506s  Loss=29142.03 [1030.83    0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_jax
L-BFGS-B        Time= 0.7937s  Loss=27745.82 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
L-BFGS-B        Time= 2.6906s  Loss=21236.60 [6.43e+02 2.77e+00 6.43e+01 2.43e+02 9.94e-01 3.53e+03 1.30e+02 1.70e+01 1.89e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


float32

single_exp_jax
L-BFGS-B        Time= 0.7216s  Loss=29154.92 [1030.9     0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_jax
L-BFGS-B        Time= 0.5401s  Loss=28268.69 [1.01e+03 0.00e+00 2.90e-01 3.60e+03 2.23e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
L-BFGS-B        Time= 0.6435s  Loss=22573.69 [1.06e+03 5.78e-01 4.31e+00 0.00e+00 7.51e-01 3.60e+03 1.89e+02 5.32e+01 2.00e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR

#### tc_bright

In [23]:
tic = -time()
F0, res = tc_brightfit(trace, np.arange(len(trace)) / frame_rate, M=TukeyBiweight(3), skewness_factor=0)
tic += time()
with np.printoptions(precision=2, suppress=False, linewidth=120):
    print(f"{'bright':14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)

bright          Time= 4.4095s  Loss=21887.81 [1.11e+03 2.62e-13 1.55e+01 7.63e+01 9.89e-01 1.24e+04 1.24e+02 1.46e+01 6.88e+02] Optimization terminated successfully.


#### scale params to all be O(1)

In [24]:
def bright_scaled(params, t, b_inf_scale: float = 1000.0, return_jac: bool = False):
    """
    Scaled wrapper for bright model.

    Parameters
    ----------
    params : np.ndarray
        [b_inf_scaled, b_slow, b_fast, b_rapid, b_bright,
         log_t_slow, log_t_fast, log_t_rapid, log_t_bright]
    t : np.ndarray
        Time vector
    b_inf_scale : float
        Scaling factor for b_inf
    return_jac : bool
        If True, return Jacobian in optimizer parameter space
    """

    (
        b_inf_scaled,
        b_slow,
        b_fast,
        b_rapid,
        b_bright,
        log_t_slow,
        log_t_fast,
        log_t_rapid,
        log_t_bright,
    ) = params

    # transform parameters
    b_inf = b_inf_scaled * b_inf_scale
    t_slow = np.exp(log_t_slow)
    t_fast = np.exp(log_t_fast)
    t_rapid = np.exp(log_t_rapid)
    t_bright = np.exp(log_t_bright)

    phys_params = (
        b_inf,
        b_slow,
        b_fast,
        b_rapid,
        b_bright,
        t_slow,
        t_fast,
        t_rapid,
        t_bright,
    )

    # evaluate model
    if not return_jac:
        return bright(phys_params, t)

    y, J_phys = bright_with_jac(phys_params, t, return_jac=True)

    # chain rule transform
    J = np.empty_like(J_phys)

    # amplitudes
    J[:, 0] = J_phys[:, 0] * b_inf_scale
    J[:, 1] = J_phys[:, 1]
    J[:, 2] = J_phys[:, 2]
    J[:, 3] = J_phys[:, 3]
    J[:, 4] = J_phys[:, 4]

    # log-time constants
    J[:, 5] = J_phys[:, 5] * t_slow
    J[:, 6] = J_phys[:, 6] * t_fast
    J[:, 7] = J_phys[:, 7] * t_rapid
    J[:, 8] = J_phys[:, 8] * t_bright

    return y, J


tic = -time()

b_inf0 = trace[-1000:].mean()
b_inf_scale = 1000.0


def fit_baseline_trend_scaled(
    trace: np.ndarray,
    t: np.ndarray,
    model: callable,
    init_params: np.ndarray,
    bounds: tuple[tuple[float, float], ...] | None = None,
    M: RobustNorm | None = None,
    optimizer: str = "L-BFGS-B",
    maxiter: int = 5,
    optimizer_options: dict | None = None,
):

    b_inf_scale = 1000.0
    
    init_params_scaled = init_params.copy()
    init_params_scaled[0] /= b_inf_scale
    init_params_scaled[5:] = np.log(init_params_scaled[5:])
    
    bounds = np.array([(0, np.inf)] * 5 + [(300, np.inf), (1, 1200), (1, 180), (60, np.inf)])
    bounds_scaled = bounds.copy()
    bounds_scaled[5:] = np.log(bounds_scaled[5:])
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        F0, res = fit_baseline_trend2(
            trace,
            timestamps,
            model=bright_scaled,  # lambda p, t: bright_scaled(np.array(p), t, b_inf_scale=b_inf_scale),
            init_params=init_params_scaled,
            bounds=bounds_scaled,
            M=M,
            optimizer=optimizer,
            maxiter=maxiter,
            optimizer_options=optimizer_options,
        )
    
    res.x[0] *= b_inf_scale        # rescale b_inf
    res.x[5:] = np.exp(res.x[5:])  # log → time constants

    return F0, res

In [25]:
tic = -time()
F0, res = fit_baseline_trend_scaled(trace, timestamps, model, bright_init, bounds=bright_bounds,
                                    M=TukeyBiweight(3),
                                    optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
tic += time()
with np.printoptions(precision=2, suppress=False, linewidth=120):
    print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)

L-BFGS-B        Time=20.3007s  Loss=21685.56 [3.51e+03 9.71e+01 8.39e+02 1.41e+02 1.00e+00 2.89e+03 1.58e+02 1.84e+01 3.42e+05] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


#### multiple traces in parallel

In [26]:
for model, jac, start_params, bounds in (
    (single_exp, single_exp_jac, single_exp_init, single_exp_bounds),
    (double_exp, double_exp_jac, double_exp_init, double_exp_bounds),
    (bright, bright_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    n_jobs = min(len(traces[0]), int(os.getenv("CO_CPUS", -1)))
    for method in ("L-BFGS-B_noJac", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            res = Parallel(n_jobs=n_jobs)(
                delayed(fit_baseline_trend)(
                    t, timestamps, model, start_params, bounds=bounds,
                    M=TukeyBiweight(3),
                    jac=jac if method=="L-BFGS-B" else None,
                    optimizer=optimizer,
                    optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10) if optimizer=="L-BFGS-B" else None)
                for t in traces[0]
            )
        tic += time()
        with np.printoptions(precision=5, suppress=False, linewidth=120):
            nl = np.mean([np.mean(r[1].fun/t.size*r[1].sigma**2 / t.var()) for r, t in zip(res, traces[0])])
            print(f"{method:14}  Time={tic:8.4f}s  Norm.Loss={nl:.5f}")


single_exp
L-BFGS-B_noJac  Time=  7.1684s  Norm.Loss=0.29393
L-BFGS-B        Time= 24.1364s  Norm.Loss=0.29344

double_exp
L-BFGS-B_noJac  Time= 13.3527s  Norm.Loss=0.28991
L-BFGS-B        Time= 20.5630s  Norm.Loss=0.29041

bright
L-BFGS-B_noJac  Time=132.1892s  Norm.Loss=0.23358
L-BFGS-B        Time= 98.3357s  Norm.Loss=0.23338


In [27]:
for model, start_params, bounds in (
    (single_exp_with_jac, single_exp_init, single_exp_bounds),
    (double_exp_with_jac, double_exp_init, double_exp_bounds),
    (bright_with_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    n_jobs = min(len(traces[0]), int(os.getenv("CO_CPUS", -1)))
    for optimizer in ("L-BFGS-B",):
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            res = Parallel(n_jobs=n_jobs)(
                delayed(fit_baseline_trend2)(
                    t, timestamps, model, start_params, bounds=bounds,
                    M=TukeyBiweight(3),
                    optimizer=optimizer,
                    optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10))
                for t in traces[0]
            )
        tic += time()
        with np.printoptions(precision=5, suppress=False, linewidth=120):
            nl = np.mean([np.mean(r[1].fun/t.size*r[1].sigma**2 / t.var()) for r, t in zip(res, traces[0])])
            print(f"{method:14}  Time={tic:8.4f}s  Norm.Loss={nl:.5f}")


single_exp_with_jac
L-BFGS-B        Time= 14.7435s  Norm.Loss=0.29344

double_exp_with_jac
L-BFGS-B        Time= 13.5318s  Norm.Loss=0.29041

bright_with_jac
L-BFGS-B        Time=106.1906s  Norm.Loss=0.23338


In [28]:
for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    
    def fit_baseline_worker(trace):
        # Re-enable float64 in this worker process
        jax.config.update("jax_enable_x64", True)
        # Call your existing function
        return fit_baseline_trend(
            trace,
            timestamps,
            model,
            start_params,
            bounds=bounds,
            M=TukeyBiweight_jax(3),
            jac="jax",
            optimizer=optimizer,
            optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10),
            dtype=dtype,
        )

    for model, start_params, bounds in (
        (single_exp_jax, single_exp_init, single_exp_bounds),
        (double_exp_jax, double_exp_init, double_exp_bounds),
        (bright_jax, bright_init, bright_bounds),
    ):
        print("\n" + model.__name__)
        n_jobs = min(len(traces[0]), int(os.getenv("CO_CPUS", -1)))
        for optimizer in ("L-BFGS-B",):
            tic = -time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                res = Parallel(n_jobs=n_jobs)(
                    delayed(fit_baseline_worker)(t) for t in traces[0]
                )
            tic += time()
            with np.printoptions(precision=5, suppress=False, linewidth=120):
                nl = np.mean([np.mean(r[1].fun/t.size*r[1].sigma**2 / t.var()) for r, t in zip(res, traces[0])])
                print(f"{optimizer:14}  Time={tic:8.4f}s  Norm.Loss={nl:.5f}") 



float64

single_exp_jax
L-BFGS-B        Time= 17.3236s  Norm.Loss=0.29184

double_exp_jax
L-BFGS-B        Time= 23.7913s  Norm.Loss=0.28916

bright_jax


/opt/conda/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


L-BFGS-B        Time= 59.2292s  Norm.Loss=0.22831


float32

single_exp_jax


/opt/conda/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


L-BFGS-B        Time= 11.0383s  Norm.Loss=0.28812

double_exp_jax
L-BFGS-B        Time= 11.9520s  Norm.Loss=0.28384

bright_jax
L-BFGS-B        Time= 17.9924s  Norm.Loss=0.23156
